In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
MODEL_ID='moonshotai/kimi-k2-instruct-0905'
assert os.environ["GROQ_API_KEY"].startswith("gsk"), \
  "Please specify the GROQ_API_KEY access token in keys.env file"
 

In [3]:
import gutenberg_text_loader as gtl
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [4]:
gs = gtl.GutenbergSource()
doc = gs.load_from_url("https://www.gutenberg.org/cache/epub/46976/pg46976.txt")

2026-04-06 11:21:36,057 - INFO - Loading https://www.gutenberg.org/cache/epub/46976/pg46976.txt from cache
2026-04-06 11:21:36,143 - INFO - Cleaned Gutenberg text: removed 1172 chars from start, 18457 chars from end
2026-04-06 11:21:36,145 - INFO - Successfully loaded text from https://www.gutenberg.org/cache/epub/46976/pg46976.txt.


In [5]:
doc.text[21000:22000]

'he calls himself so in _Cynegeticus_ (v.\n6); and in _Periplus_ (xii. 5; xxv. 1), he distinguishes Xenophon by\nthe addition _the elder_. Lucian (_Alexander_, 56) calls Arrian simply\n_Xenophon_. During the stay of the emperor Hadrian at Athens, A.D. 126,\nArrian gained his friendship. He accompanied his patron to Rome, where\nhe received the Roman citizenship. In consequence of this, he assumed\nthe name of Flavius.[2] In the same way the Jewish historian, Josephus,\nhad been allowed by Vespasian and Titus to bear the imperial name\nFlavius.[3]\n\nPhotius says, that Arrian had a distinguished career in Rome, being\nentrusted with various political offices, and at last reaching the\nsupreme dignity of consul under Antoninus Pius.[4] Previous to this\nhe was appointed (A.D. 132) by Hadrian, Governor of Cappadocia, which\nprovince was soon after invaded by the Alani, or Massagetae, whom he\ndefeated and expelled.[5] When Marcus Aurelius came to the throne,\nArrian withdrew into private 

In [6]:
print(doc.id_)

b19aa8a2-f7b4-4cb5-878e-edf8213244b0


In [7]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import Document

class Indexer:
    """
    A class to load documents into LlamaIndex using BM25
    Attributes:
      chunk_size (int): Size of text chunks for processing.
      chunk_overlap (int): Overlap between text chunks.
      docstore (SimpleDocumentStore): Document store for storing processed documents.
    """

    def __init__(self,
                 cache_dir: str = './.cache',
                 chunk_size = 1024,
                 chunk_overlap = 20):
        """
        Initialize the Indexer

        Args:
          chunk_size(int): Size of text chunks for processing. Defaults to 1024.
          chunk_overlap (int): Overlap between text chunks. Defaults to 20.
        """

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        self.docstore = SimpleDocumentStore()
        self.node_parser = SentenceSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap
        )

        logger.info("Indexer initialized")

    def add_document_to_index(self, document: Document):
        nodes = self.node_parser.get_nodes_from_documents([document])
        self.docstore.add_documents(nodes)
        logger.info(f"Successfully loaded text from {document.id_} -- {len(nodes)} created")
    
    def get_docstore(self) -> SimpleDocumentStore:
        return self.docstore

In [8]:
index = Indexer(chunk_size=100, chunk_overlap=20)
index.add_document_to_index(doc)

2026-04-06 11:21:36,992 - INFO - Indexer initialized
2026-04-06 11:21:40,878 - INFO - Successfully loaded text from b19aa8a2-f7b4-4cb5-878e-edf8213244b0 -- 6104 created


In [9]:
from llama_index.retrievers.bm25 import BM25Retriever
retriever = BM25Retriever.from_defaults(
    docstore=index.get_docstore(),
    similarity_top_k=5
)

2026-04-06 11:21:42,638 - DEBUG - Building index from IDs objects


In [10]:
from llama_index.core.response.notebook_utils import display_source_node
retrieved_nodes = retriever.retrieve("Describe the relationship between Alexander and Diogenes")
for node in retrieved_nodes:
    display_source_node(node, 1024)

**Node ID:** c7efbd1c-021f-4ae2-bd40-a075dcdfec21<br>**Similarity:** 4.235982418060303<br>**Text:** But Diogenes said that he
wanted nothing else, except that he and his attendants would stand out
of the sunlight. Alexander is said to have expressed his admiration
of Diogenes’s conduct.<br>

**Node ID:** 2fc44d5e-546a-427f-9c30-83b7fd35f3d8<br>**Similarity:** 4.107966899871826<br>**Text:** 100 stades; and most of it is the mean between
these breadths.[642] This river Indus Alexander crossed at daybreak
with his army into the country of the Indians; concerning whom, in
this history I have described neither what laws they enjoy,<br>

**Node ID:** 7a9555bf-4df4-447c-9c9a-38240c5f2f90<br>**Similarity:** 3.6308696269989014<br>**Text:** 32). Alexander said: “If I were
not Alexander, I should like to be Diogenes.” Cf. _Arrian_, i. 1;
Plutarch (_de Fortit. Alex._, p. 331).<br>

**Node ID:** ccc82562-e5ab-4c72-97a9-67f778e9e115<br>**Similarity:** 3.4012913703918457<br>**Text:** Alexander is said to have expressed his admiration
of Diogenes’s conduct.[832] Thus it is evident that Alexander was
not entirely destitute of better feelings; but he was the slave of
his insatiable ambition.<br>

**Node ID:** b780cff6-9b70-4fa3-b86b-9baa588a98a2<br>**Similarity:** 3.2427611351013184<br>**Text:** He also ascertained that for
the present Bessus held the supreme command, both on account of his
relationship to Darius and because the war was being carried on in his
viceregal province. Hearing this,<br>

In [11]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.groq import Groq

llm = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
    model=MODEL_ID,
    temperature=0.2)


In [12]:
from llama_index.core.llms import ChatMessage
messages = [
    ChatMessage(
        role="system",
        content="Use the following text to answer the given question."
    )
]

messages += [
    ChatMessage(role="system", content=node.text) for node in retrieved_nodes
]

messages += [
    ChatMessage(role="user", content="Describe the relationship between Alexander and Diogenes.")
]

response = llm.chat(messages)
print(response)

2026-04-06 11:22:15,908 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


assistant: Alexander and Diogenes shared a relationship of mutual, philosophical recognition rather than personal friendship.  Alexander admired Diogenes’s conduct and, according to tradition, said: “If I were not Alexander, I should like to be Diogenes.”  Diogenes, in turn, acknowledged Alexander’s presence only to the extent of asking him and his attendants to stand out of the sunlight.  Their interaction was brief, symbolic, and entirely public; no private acquaintance existed.


In [13]:
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever, llm=llm
)

response = query_engine.query("Describe the relationship between Alexander and Diogenes.")
response = {
    "answer": str(response),
    "source_nodes": response.source_nodes
}
print(response['answer'])

2026-04-06 11:22:16,418 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander admired Diogenes. When Diogenes asked only that Alexander and his attendants stand out of the sunlight, Alexander expressed admiration for Diogenes’s conduct. Alexander even remarked that if he were not Alexander, he would like to be Diogenes.


In [14]:
for node in response['source_nodes']:
    print(node)

Node ID: c7efbd1c-021f-4ae2-bd40-a075dcdfec21
Text: But Diogenes said that he wanted nothing else, except that he
and his attendants would stand out of the sunlight. Alexander is said
to have expressed his admiration of Diogenes’s conduct.
Score:  4.236

Node ID: 2fc44d5e-546a-427f-9c30-83b7fd35f3d8
Text: 100 stades; and most of it is the mean between these
breadths.[642] This river Indus Alexander crossed at daybreak with his
army into the country of the Indians; concerning whom, in this history
I have described neither what laws they enjoy,
Score:  4.108

Node ID: 7a9555bf-4df4-447c-9c9a-38240c5f2f90
Text: 32). Alexander said: “If I were not Alexander, I should like to
be Diogenes.” Cf. _Arrian_, i. 1; Plutarch (_de Fortit. Alex._, p.
331).
Score:  3.631

Node ID: ccc82562-e5ab-4c72-97a9-67f778e9e115
Text: Alexander is said to have expressed his admiration of Diogenes’s
conduct.[832] Thus it is evident that Alexander was not entirely
destitute of better feelings; but he was the slave

In [17]:
def build_query_engine(urls: list[str], chunk_size: int) -> RetrieverQueryEngine:
    gs = gtl.GutenbergSource()
    index = Indexer(chunk_size=chunk_size, chunk_overlap=chunk_size//10)

    for url in urls:
        doc = gs.load_from_url(url)
        index.add_document_to_index(doc)

    retriever = BM25Retriever.from_defaults(
        docstore=index.docstore,
        similarity_top_k=5
    )

    llm = Groq(model=MODEL_ID,
               api_key= os.environ['GROQ_API_KEY'],
               temperature=0.2
               )
    
    query_engine = RetrieverQueryEngine.from_args(
        retriever=retriever, llm=llm
    )

    return query_engine

def print_response_to_query(query_engine: RetrieverQueryEngine, query: str):
    response = query_engine.query(query)
    response = {
        "answer": str(response),
          "source_nodes": response.source_nodes  
          }
    print(response['answer'])
    print('\n\n**Sources**:')
    for node in response['source_nodes']:
        print(node)
    

In [18]:
query_engine = build_query_engine(["https://www.gutenberg.org/files/53669/53669-0.txt"], 100)
print_response_to_query(query_engine, "What should I do if the diaphragm is ruptured")

2026-04-06 11:30:41,158 - INFO - Indexer initialized
2026-04-06 11:30:41,161 - INFO - Loading https://www.gutenberg.org/files/53669/53669-0.txt from cache
2026-04-06 11:30:41,170 - INFO - Cleaned Gutenberg text: removed 50 chars from start, 49 chars from end
2026-04-06 11:30:41,171 - INFO - Successfully loaded text from https://www.gutenberg.org/files/53669/53669-0.txt.
2026-04-06 11:30:41,738 - INFO - Successfully loaded text from 3c9395fa-84c6-46ac-a2cd-43af571a664e -- 1208 created
2026-04-06 11:30:41,930 - DEBUG - Building index from IDs objects
2026-04-06 11:30:42,651 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Replace the safety head with an unbroken one.


**Sources**:
Node ID: fe294d10-a711-401b-9dc4-9b971f5ecf06
Text: Inspect to see if diaphragm is intact. If diaphragm is ruptured,
replace the safety head with an unbroken head.
Score:  4.868

Node ID: 5306de5d-970f-44f5-928d-dcbeb3689155
Text: (3) Unscrew diaphragm cap and pull out washer, support, and
valve-diaphragm assembly. To prevent loss of valve-needle adjustment
(Fig 54), do not disturb position of yoke block by turning the needle.
Score:  3.281

Node ID: 378d33d3-33cd-4762-9098-0903e1e7a8f5
Text: (Fig 52) Screw on the diaphragm cap by hand. Do not use a
wrench.   Install valve grip. (Par 74 _c_)    (4) Place valve spring
over end of needle and install spring   retainer.
Score:  2.675

Node ID: b08cd81f-8998-41de-8d33-4fc62107a98e
Text: If the diaphragm   shows evidence of tears or separation, or if
leaks occur at the   diaphragm, replace the valve-diaphragm assembly.
Score:  2.472

Node ID: 6f8817fc-5b68-490b-b9c8-b205ed6da562
Te

In [19]:
print_response_to_query(query_engine, "What should I do if diaphragm is broken")

2026-04-06 11:32:19,622 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Unscrew the diaphragm cap, remove the washer, support, and the damaged valve-diaphragm assembly, then install a new diaphragm and reassemble the cap by hand.


**Sources**:
Node ID: 5306de5d-970f-44f5-928d-dcbeb3689155
Text: (3) Unscrew diaphragm cap and pull out washer, support, and
valve-diaphragm assembly. To prevent loss of valve-needle adjustment
(Fig 54), do not disturb position of yoke block by turning the needle.
Score:  3.281

Node ID: 755779a5-f71d-44db-a9a2-5951915ece53
Text: (Par 49)    (2) _Spring-case assembly._ If outer case rotates
and inner case does   not, and no spring action occurs, spring is
broken and spring case   should be replaced as a unit.
Score:  2.702

Node ID: 378d33d3-33cd-4762-9098-0903e1e7a8f5
Text: (Fig 52) Screw on the diaphragm cap by hand. Do not use a
wrench.   Install valve grip. (Par 74 _c_)    (4) Place valve spring
over end of needle and install spring   retainer.
Score:  2.675

Node ID: f49eee6e-0a99-4578-a27c-85a5a7d734e5
Text: If end of trig

In [20]:
def print_response(chunk_size: int)-> str:
    query_engine = build_query_engine(["https://www.gutenberg.org/files/53669/53669-0.txt"], chunk_size=chunk_size)
    response = query_engine.query("What should I do if diaphragm is ruptured?")
    print(response)
print_response(100)

2026-04-06 11:35:10,920 - INFO - Indexer initialized
2026-04-06 11:35:10,923 - INFO - Loading https://www.gutenberg.org/files/53669/53669-0.txt from cache
2026-04-06 11:35:10,928 - INFO - Cleaned Gutenberg text: removed 50 chars from start, 49 chars from end
2026-04-06 11:35:10,931 - INFO - Successfully loaded text from https://www.gutenberg.org/files/53669/53669-0.txt.
2026-04-06 11:35:11,545 - INFO - Successfully loaded text from 02c72e23-e735-4f02-82cf-4444b44e5333 -- 1208 created
2026-04-06 11:35:12,463 - DEBUG - Building index from IDs objects
2026-04-06 11:35:13,125 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Replace the safety head with an unbroken head.


In [23]:
print_response(200)

2026-04-06 11:36:24,160 - INFO - Indexer initialized
2026-04-06 11:36:24,163 - INFO - Loading https://www.gutenberg.org/files/53669/53669-0.txt from cache
2026-04-06 11:36:24,177 - INFO - Cleaned Gutenberg text: removed 50 chars from start, 49 chars from end
2026-04-06 11:36:24,178 - INFO - Successfully loaded text from https://www.gutenberg.org/files/53669/53669-0.txt.
2026-04-06 11:36:24,503 - INFO - Successfully loaded text from 729708a7-570a-4140-8b22-950142b32938 -- 376 created
2026-04-06 11:36:24,774 - DEBUG - Building index from IDs objects
2026-04-06 11:36:25,534 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Replace the safety head with an unbroken one.


In [24]:
print_response(500)

2026-04-06 11:36:27,961 - INFO - Indexer initialized
2026-04-06 11:36:27,963 - INFO - Loading https://www.gutenberg.org/files/53669/53669-0.txt from cache
2026-04-06 11:36:27,968 - INFO - Cleaned Gutenberg text: removed 50 chars from start, 49 chars from end
2026-04-06 11:36:27,972 - INFO - Successfully loaded text from https://www.gutenberg.org/files/53669/53669-0.txt.
2026-04-06 11:36:28,251 - INFO - Successfully loaded text from b58a83ee-428d-47ba-b096-f2d1d074d9ad -- 124 created
2026-04-06 11:36:28,365 - DEBUG - Building index from IDs objects
2026-04-06 11:36:29,248 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Replace the safety head with an unbroken one.
